# Porównanie planów spłaty kredytu studenckiego za pomocą PROC LOAN

## Streszczenie

Biuro pomocy finansowej ocenia, jak absolwencki rocznik powinien spłacać reprezentatywne saldo w wysokości $27,500, porównując cztery struktury spłaty — federalny plan standardowy o stałym oprocentowaniu, prywatne refinansowanie z prowizją za udzielenie, kredyt o zmiennym oprocentowaniu z limitem (ARM) oraz dopłatę sponsorowaną przez pracodawcę — za pomocą `PROC LOAN`. W ciągu 120-miesięcznego okresu cztery oferty układają się jednoznacznie pod względem miesięcznej raty i odsetek całkowitych przy podanych stopach początkowych: plan federalny standardowy kosztuje najwięcej odsetek (**$10,021.22** przy 6.53%, rata **$312.68**), prywatne refinansowanie obniża oprocentowanie, ale dodaje prowizję $412.50 (**$9,120.20** przy 5.99%, rata **$305.17**), a niżej oprocentowany ARM (**$7,099.76** przy 4.75%, rata **$288.33**) oraz dopłata pracodawcy (**$6,700.67** przy 4.50%, rata **$285.01**) mają najniższe koszty odsetkowe. Blok `COMPARE` raportuje następnie skumulowane odsetki, kapitał i saldo pozostające do spłaty dla każdego planu w miesiącach 36, 60 i 120, pokazując, jak daleko zaawansowana jest amortyzacja każdego kredytu w momentach, w których kredytobiorca najprawdopodobniej dokona refinansowania lub spłaci kredyt.

## Źródła danych

| Zbiór danych | Wiersze | Opis | Kluczowe zmienne |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Syntetyczne profile kredytowe absolwenckiego rocznika, wygenerowane bezpośrednio za pomocą `call streaminit(20260531)` i `rand('uniform')`. Służą do umotywowania realistycznych warunków kredytu do porównania. | `student_id` (1001-1040), `balance` (kapitał w momencie ukończenia studiów; ten losowy zestaw obejmuje $11,800-$47,750, średnia $30,771), `apr` (nominalne roczne oprocentowanie 4.10%-9.10%, średnia 6.50%), `term` (120 lub 180 miesięcy, średnia 144), `origination_pct` (prowizja 1.0%-2.0%, średnia 1.50%) |

Reprezentatywny kredytobiorca analizowany za pomocą `PROC LOAN` (kwota $27,500, okres 120 miesięcy, start lipiec 2026) plasuje się blisko dolnego środka rozkładu salda i oprocentowania tego rocznika; nie są używane żadne dane zewnętrzne ani sieciowe. Rocznik istnieje, aby umotywować realistyczne warunki kredytu — bezpośrednie porównanie jest wykonywane na pojedynczym reprezentatywnym kredycie.

# Porównanie planów spłaty kredytu studenckiego za pomocą PROC LOAN

Gdy studenci kończą studia, biuro pomocy finansowej musi pomóc im wybrać spośród konkurujących ze sobą ofert spłaty. `PROC LOAN` (SAS/ETS) jest stworzony dokładnie do tego celu: amortyzuje kredyty o stałym oprocentowaniu, o zmiennym oprocentowaniu (ARM) oraz z dopłatą (buydown), a następnie porównuje je obok siebie pod względem raty, odsetek całkowitych i postępu amortyzacji.

W tym notatniku:

1. Generujemy syntetyczny rocznik absolwentów, aby ustalić realistyczne warunki kredytu.
2. Podsumowujemy rocznik za pomocą `PROC MEANS`.
3. Budujemy cztery plany spłaty dla reprezentatywnego salda $27,500 i porównujemy je za pomocą `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. Ponownie uruchamiamy zalecany plan o stałym oprocentowaniu samodzielnie, aby potwierdzić jego niezależną ekonomikę.

Wszystko działa lokalnie na wewnętrznie generowanych danych syntetycznych — bez plików zewnętrznych ani dostępu do sieci.

## Krok 1 — Generowanie syntetycznego rocznika absolwentów

Symulujemy 40 kredytobiorców. Każdy otrzymuje kapitał w momencie ukończenia studiów, RRSO luźno powiązane z jakością kredytową, standardowy okres spłaty (10 lub 15 lat) oraz procentową prowizję za udzielenie kredytu. Ziarno losowania sprawia, że dane są odtwarzalne.

In [1]:
DANE borrowers;
   CALL streaminit(20260531);
   DŁUGOŚĆ plan $ 28;
   POWTÓRZ student_id = 1001 TO 1040;
      /* Kapitał w momencie ukończenia studiów: $9,500 - $48,000 */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Nominalne roczne RRSO wg kategorii zdolności kredytowej: 4.0% - 9.5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Standardowy okres spłaty: 120 lub 180 miesięcy */
      JEŚLI rand('uniform') < 0.6 WTEDY term = 120;
      PRZECIWNIE term = 180;
      /* Prowizja za udzielenie kredytu jako procent kapitału: 1.0% - 2.0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      WYJŚCIE;
   KONIEC;
WYKONAJ;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Krok 2 — Profilowanie rocznika

Zanim zamodelujemy poszczególne kredyty, przyjrzyjmy się rozkładowi sald, oprocentowań i okresów spłaty. To pokazuje, jak wygląda *reprezentatywny* kredytobiorca — podstawę bezpośredniego porównania, które następuje dalej.

In [2]:
PROCEDURA ŚREDNIE DANE=borrowers n mean MIN MAX maxdec=2;
   ETYKIETA balance='Saldo ($)' apr='RRSO (%)' term='Okres (miesiące)' origination_pct='Prowizja za udzielenie (%)';
   ZMIENNA balance apr term origination_pct;
WYKONAJ;

                                                  The MEANS Procedure

 Variable         Label                             N           Mean     Minimum     Maximum
 -------------------------------------------------------------------------------------------
 balance          Saldo ($)                        40       30771.25    11800.00    47750.00
 apr              RRSO (%)                         40           6.50        4.10        9.10
 term             Okres (miesiące)                 40         144.00      120.00      180.00
 origination_pct  Prowizja za udzielenie (%)       40           1.50        1.00        2.00
 -------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 3 — Porównanie czterech planów spłaty dla reprezentatywnego kredytobiorcy

Używając reprezentatywnego salda $27,500 w okresie 120 miesięcy (10 lat) rozpoczynającym się w lipcu 2026, zestawiamy cztery realistyczne oferty:

- **Kredyt Federalny Standardowy (bez dopłat)** — zwykły kredyt o stałym oprocentowaniu 6.53%.
- **Refinansowanie prywatne (z opłatą)** — niższe stałe oprocentowanie 5.99%, ale z kosztem inicjalizacji $412.50 (`INIT=`).
- **Prywatny kredyt o zmiennym oprocentowaniu ARM** — kredyt regulowany o oprocentowaniu 4.75%, z limitem 1% na korektę / 5% w całym okresie (`CAPS=`), `MAXRATE=` na poziomie 9.75%, roczną częstotliwością korekt `ADJUSTFREQ=` oraz słowem kluczowym `WORSTCASE`.
- **Dopłata pracodawcy 2-1** — dofinansowany start 4.50%, który zgodnie z podanym harmonogramem wzrasta poprzez `BDRATES=` do 5.50% w drugim roku i do 6.50% później.

Instrukcja `COMPARE` żąda widoku porównawczego między kredytami w miesiącach 36, 60 i 120, z `TAXRATE=` na poziomie 22% oraz minimalną atrakcyjną stopą zwrotu 4% (`MARR=`); `OUTSUM=` i `OUTCOMP=` przechwytują podsumowanie dla każdego kredytu oraz wiersze porównania. Poniższy wydruk raportuje dla każdego planu i każdego punktu kontrolnego **skumulowane zapłacone odsetki, skumulowany spłacony kapitał oraz saldo pozostające do spłaty** — czytelny widok postępu amortyzacji wśród kandydatów.

> **Uwaga o korektach oprocentowania.** `PROC LOAN` w Jennerze amortyzuje każdy plan według jego podanej stopy **początkowej**, więc ustawienia `CAPS`/`MAXRATE`/`WORSTCASE` kredytu ARM oraz kroki `BDRATES` dopłaty pracodawcy są odzwierciedlone na wydruku jako warunki kredytu, ale **nie** są stosowane w matematyce raty — poniższe wartości dla ARM i dopłaty odzwierciedlają ich stopy początkowe 4.75% i 4.50% utrzymywane bez zmian przez cały okres. Traktuj te dwie sumy jako koszty w najlepszym przypadku (przy stopie początkowej), a nie jako pułapy najgorszego scenariusza.

In [3]:
PROCEDURA loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label='Kredyt Federalny Standardowy';

   fixed rate=5.99 init=412.50
         label='Refinansowanie (z opłatą)';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label='Prywatny ARM (najgorszy)';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label='Dopłata pracodawcy 2-1';

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
WYKONAJ;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Kredyt Federalny Standardowy
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Refinansowanie (z opłatą)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: Prywatny ARM (najgorszy)
    Loan Type:                   Adjustable Rate
    Amount (Principal):                27500.00
   


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Krok 4 — Ponowne uruchomienie zalecanego planu o stałym oprocentowaniu samodzielnie

Dla kredytobiorcy, który ceni pewność raty, federalny plan standardowy o stałym oprocentowaniu jest bezpieczną opcją domyślną. Uruchamiamy go ponownie w izolacji, aby potwierdzić jego kluczową ekonomikę: tę samą miesięczną ratę **$312.68**, **$37,521.22** łącznie zapłacone i **$10,021.22** odsetek całkowitych widziane w porównaniu czterech opcji, teraz przedstawione jako samodzielne podsumowanie kredytu.

In [4]:
PROCEDURA loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label='Kredyt Federalny Standardowy';
WYKONAJ;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Kredyt Federalny Standardowy
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Kredyt Federalny Standardowy
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4         21035.90        3752.12 


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Interpretacja wyników

Wszystkie cztery plany w pełni amortyzują ten sam kapitał $27,500 w ciągu 120 miesięcy (saldo pozostające do spłaty każdego planu osiąga $0.00 w 120. miesiącu w tabeli COMPARE), więc decyzja sprowadza się do **miesięcznej raty** i **odsetek całkowitych przy podanej stopie**:

| Plan | Oprocentowanie | Rata | Odsetki całkowite | Uwagi |
|------|------|---------|----------------|-------|
| Kredyt Federalny Standardowy | 6.53% | $312.68 | $10,021.22 | Bez opłaty; najsilniejsza ochrona kredytobiorcy |
| Refinansowanie prywatne | 5.99% | $305.17 | $9,120.20 | Prowizja za udzielenie $412.50 |
| Prywatny kredyt ARM | 4.75% (początkowe) | $288.33 | $7,099.76 | Oprocentowanie może wzrosnąć; kwota to koszt przy stopie początkowej |
| Dopłata pracodawcy 2-1 | 4.50% (początkowe) | $285.01 | $6,700.67 | Zależy od kontynuacji zatrudnienia |

- Plan **federalny standardowy** jest najdroższy pod względem odsetek ($10,021.22), ale oferuje stałą, przewidywalną ratę $312.68 bez opłaty.
- **Refinansowanie prywatne** obniża oprocentowanie do 5.99% ($9,120.20 odsetek, o $901 mniej niż plan federalny), ale pobiera prowizję za udzielenie $412.50, więc jego przewaga netto nad planem federalnym wynosi w przybliżeniu $488 odsetek minus opłata — istotna tylko wtedy, gdy kredyt trwa wystarczająco długo, aby nie zostać wcześniej zrefinansowanym.
- **ARM** i **dopłata** wykazują tutaj najniższe odsetki ($7,099.76 i $6,700.67) wyłącznie dlatego, że ich stopy **początkowe** są znacznie niższe od ofert stałych. Jak zaznaczono w Kroku 3, Jenner utrzymuje te stopy początkowe bez zmian, więc są to wartości najlepszego przypadku: rzeczywisty ARM, który koryguje oprocentowanie w górę — lub dopłata, która traci dofinansowanie pracodawcy — wypadłyby wyżej, a rata nie jest gwarantowana.

Tabela `COMPARE` doprecyzowuje to, pokazując, jak szybko amortyzuje się każdy plan. Po 36 miesiącach plan federalny zapłacił $4,792.27 odsetek i spłacił $6,464.10 kapitału (saldo $21,035.90), podczas gdy dopłata zapłaciła tylko $3,263.97 odsetek i spłaciła $6,996.24 kapitału (saldo $20,503.76) — plany o niższym oprocentowaniu kosztują mniej odsetek i szybciej redukują kapitał w ciągu pierwszych trzech lat. Dla absolwenta unikającego ryzyka pewność oprocentowania planu federalnego standardowego często uzasadnia jego wyższe odsetki; dla kredytobiorcy pewnego stabilnego, trwałego zatrudnienia dofinansowany start dopłaty pracodawcy zapewnia największe oszczędności spośród opcji, które faktycznie ustalają stałą stopę.